# このnotebookでは

KVキャッシュが何を保存し、なぜ自己回帰生成を速くするのかを、概念説明と最小実装の両方で理解する。

結論:

KVキャッシュは、過去トークンから計算済みの `K` と `V` を保存して再利用する仕組みであり、生成ステップごとの再計算を減らすことで推論を高速化する。速くなる代わりにメモリを使い、主に自己回帰な推論時に効く。

## このノートの見方

1. まずは `Q`, `K`, `V` の役割を整理する。
2. 次に、KVキャッシュなしの生成で何を毎回再計算しているかを見る。
3. その後、KVキャッシュありの生成に切り替え、何が省略されるかを確認する。
4. 最後に、計算量・メモリ・よくある誤解をまとめる。

このノートでは、説明を簡潔にするために `batch=1` の single-head self-attention を使う。実際の LLM では multi-head だが、本質は同じである。

In [ ]:
import math
import platform
import sys

import numpy as np
import torch

In [ ]:
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Torch:", torch.__version__)
print("NumPy:", np.__version__)
print(
    "MPS available:",
    torch.backends.mps.is_available() if hasattr(torch.backends, "mps") else False,
)
print("CUDA available:", torch.cuda.is_available())

Python: 3.13.3
Platform: macOS-26.4.1-arm64-arm-64bit-Mach-O
Torch: 2.11.0
NumPy: 2.4.3
MPS available: True
CUDA available: False


## 1. `Q`, `K`, `V` の役割

self-attention では、各トークン埋め込み `x_t` から線形変換で `Q`, `K`, `V` を作る。

- `Q` (`query`): 今見ているトークンが、過去のどこを参照したいかを表す
- `K` (`key`): 各トークンが「自分はこういう特徴を持つ」と示す手がかり
- `V` (`value`): 実際に集約される中身

1ステップ生成では、最新トークンの `Q` が過去すべての `K` を見て重みを作り、その重みで `V` の加重平均を取る。

式で書くと、1つの attention head では次になる。

`Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) V`

重要なのは、自己回帰生成では毎ステップ「最新トークン1個を追加して次を出す」ことだが、過去トークンの `K` と `V` 自体は変わらない点である。ここにキャッシュの余地がある。

## 2. このノートで使うテンソル形状

このノートでは、`batch=1` を固定し、系列方向を先頭に置く。

- 入力埋め込み `x`: `(seq_len, d_model)`
- `Q`, `K`, `V`: `(seq_len, d_model)`
- attention score: `(query_len, key_len)`
- KVキャッシュ: `K_cache`, `V_cache` ともに `(past_len, d_model)`

生成ステップで変わるのは `past_len` だけである。

In [4]:
torch.manual_seed(7)

d_model = 4
vocab = ["A", "B", "C", "D", "E", "F"]
token_to_id = {token: i for i, token in enumerate(vocab)}
id_to_token = {i: token for token, i in token_to_id.items()}

# 小さな埋め込み表と投影行列を固定する。
embedding_table = torch.randn(len(vocab), d_model)
W_q = torch.randn(d_model, d_model)
W_k = torch.randn(d_model, d_model)
W_v = torch.randn(d_model, d_model)
W_out = torch.randn(d_model, len(vocab))

prompt_tokens = ["A", "C", "B", "D"]
prompt_ids = torch.tensor([token_to_id[t] for t in prompt_tokens])
prompt_embeddings = embedding_table[prompt_ids]

print("prompt:", prompt_tokens)
print("prompt_embeddings shape:", tuple(prompt_embeddings.shape))

prompt: ['A', 'C', 'B', 'D']
prompt_embeddings shape: (4, 4)


In [ ]:
def project_qkv(x, W_q, W_k, W_v):
    Q = x @ W_q
    K = x @ W_k
    V = x @ W_v
    return Q, K, V


def causal_mask(query_len, key_len, device):
    q_positions = torch.arange(query_len, device=device).unsqueeze(1)
    k_positions = torch.arange(key_len, device=device).unsqueeze(0)
    return k_positions <= q_positions + (key_len - query_len)


def attention_from_qkv(Q, K, V, use_causal_mask=True):
    scores = Q @ K.T / math.sqrt(Q.shape[-1])
    if use_causal_mask:
        mask = causal_mask(Q.shape[0], K.shape[0], scores.device)
        scores = scores.masked_fill(~mask, float("-inf"))
    weights = torch.softmax(scores, dim=-1)
    context = weights @ V
    return context, weights


def next_token_from_context(context, W_out):
    logits = context[-1] @ W_out
    next_id = int(torch.argmax(logits).item())
    return next_id, logits

In [6]:
Q, K, V = project_qkv(prompt_embeddings, W_q, W_k, W_v)
context, weights = attention_from_qkv(Q, K, V, use_causal_mask=True)

print("Q shape:", tuple(Q.shape))
print("K shape:", tuple(K.shape))
print("V shape:", tuple(V.shape))
print("attention weights shape:", tuple(weights.shape))
print("last query attends over", weights.shape[-1], "past tokens")
weights[-1]

Q shape: (4, 4)
K shape: (4, 4)
V shape: (4, 4)
attention weights shape: (4, 4)
last query attends over 4 past tokens


tensor([0.3071, 0.0952, 0.4049, 0.1927])

## 3. KVキャッシュなし: 毎ステップで過去全体を再計算する

自己回帰生成では、次のように prefix が少しずつ伸びる。

- step 1: `A C B D`
- step 2: `A C B D ?`
- step 3: `A C B D ? ?`

キャッシュがない場合、各ステップで prefix 全体の埋め込みから `Q`, `K`, `V` を作り直す。つまり、過去の `K` と `V` も毎回再計算している。

In [ ]:
def generate_without_cache(initial_ids, steps, embedding_table, W_q, W_k, W_v, W_out):
    sequence = initial_ids.clone()
    logs = []

    for step in range(steps):
        x = embedding_table[sequence]
        Q, K, V = project_qkv(x, W_q, W_k, W_v)
        context, weights = attention_from_qkv(Q, K, V, use_causal_mask=True)
        next_id, logits = next_token_from_context(context, W_out)

        logs.append(
            {
                "step": step + 1,
                "sequence_len": int(sequence.numel()),
                "recomputed_kv_tokens": int(sequence.numel()),
                "last_query_key_count": int(weights.shape[-1]),
                "predicted_token": id_to_token[next_id],
            }
        )

        sequence = torch.cat([sequence, torch.tensor([next_id])])

    return sequence, logs


generated_without_cache, no_cache_logs = generate_without_cache(
    prompt_ids,
    steps=3,
    embedding_table=embedding_table,
    W_q=W_q,
    W_k=W_k,
    W_v=W_v,
    W_out=W_out,
)

no_cache_logs

[{'step': 1,
  'sequence_len': 4,
  'recomputed_kv_tokens': 4,
  'last_query_key_count': 4,
  'predicted_token': 'F'},
 {'step': 2,
  'sequence_len': 5,
  'recomputed_kv_tokens': 5,
  'last_query_key_count': 5,
  'predicted_token': 'E'},
 {'step': 3,
  'sequence_len': 6,
  'recomputed_kv_tokens': 6,
  'last_query_key_count': 6,
  'predicted_token': 'F'}]

## 4. KVキャッシュあり: 過去の `K`, `V` を保存して足し込む

KVキャッシュありでは、過去トークンについては `K` と `V` を保存しておく。

- 最初の prefix で `K_cache`, `V_cache` を作る
- 次のステップでは、新しく追加された token 1個からだけ `Q`, `K`, `V` を計算する
- `K_cache`, `V_cache` に新しい `K`, `V` を連結する
- 最新 token の `Q` が、キャッシュ済みの `K_cache` 全体を参照して attention を行う

つまり、毎回必要なのは「最新 token 分の投影」と「既存キャッシュとの照合」だけであり、過去 token の `K`, `V` の再生成は不要になる。

In [ ]:
def generate_with_cache(initial_ids, steps, embedding_table, W_q, W_k, W_v, W_out):
    sequence = initial_ids.clone()
    x0 = embedding_table[sequence]
    _, K_cache, V_cache = project_qkv(x0, W_q, W_k, W_v)
    logs = [
        {
            "step": 0,
            "cache_len": int(K_cache.shape[0]),
            "new_kv_tokens": int(K_cache.shape[0]),
            "note": "initial prefix is encoded once",
        }
    ]

    for step in range(steps):
        last_token_id = sequence[-1:]
        x_new = embedding_table[last_token_id]
        Q_new, K_new, V_new = project_qkv(x_new, W_q, W_k, W_v)

        context, weights = attention_from_qkv(
            Q_new, K_cache, V_cache, use_causal_mask=False
        )
        next_id, logits = next_token_from_context(context, W_out)

        next_x = embedding_table[torch.tensor([next_id])]
        _, K_next, V_next = project_qkv(next_x, W_q, W_k, W_v)
        K_cache = torch.cat([K_cache, K_next], dim=0)
        V_cache = torch.cat([V_cache, V_next], dim=0)
        sequence = torch.cat([sequence, torch.tensor([next_id])])

        logs.append(
            {
                "step": step + 1,
                "cache_len": int(K_cache.shape[0]),
                "new_kv_tokens": 1,
                "last_query_key_count": int(weights.shape[-1]),
                "predicted_token": id_to_token[next_id],
            }
        )

    return sequence, logs, K_cache, V_cache


generated_with_cache, cache_logs, K_cache_final, V_cache_final = generate_with_cache(
    prompt_ids,
    steps=3,
    embedding_table=embedding_table,
    W_q=W_q,
    W_k=W_k,
    W_v=W_v,
    W_out=W_out,
)

cache_logs

[{'step': 0,
  'cache_len': 4,
  'new_kv_tokens': 4,
  'note': 'initial prefix is encoded once'},
 {'step': 1,
  'cache_len': 5,
  'new_kv_tokens': 1,
  'last_query_key_count': 4,
  'predicted_token': 'F'},
 {'step': 2,
  'cache_len': 6,
  'new_kv_tokens': 1,
  'last_query_key_count': 5,
  'predicted_token': 'E'},
 {'step': 3,
  'cache_len': 7,
  'new_kv_tokens': 1,
  'last_query_key_count': 6,
  'predicted_token': 'F'}]

In [9]:
tokens_without_cache = [id_to_token[int(i)] for i in generated_without_cache.tolist()]
tokens_with_cache = [id_to_token[int(i)] for i in generated_with_cache.tolist()]

print("without cache:", tokens_without_cache)
print("with cache   :", tokens_with_cache)
print("same sequence:", torch.equal(generated_without_cache, generated_with_cache))
print("final K_cache shape:", tuple(K_cache_final.shape))
print("final V_cache shape:", tuple(V_cache_final.shape))

without cache: ['A', 'C', 'B', 'D', 'F', 'E', 'F']
with cache   : ['A', 'C', 'B', 'D', 'F', 'E', 'F']
same sequence: True
final K_cache shape: (7, 4)
final V_cache shape: (7, 4)


ここで出力系列が一致するのは重要である。KVキャッシュは**近似**ではなく、同じ計算結果をより少ない再計算で得るための仕組みだからである。

ただし、内部実装の順番は違う。

- キャッシュなし: prefix 全体から毎回 `Q`, `K`, `V` を作る
- キャッシュあり: prefix の過去分は `K`, `V` を使い回し、最新 token 分だけ新規に作る

In [ ]:
no_cache_recomputed = sum(log["recomputed_kv_tokens"] for log in no_cache_logs)
with_cache_new = cache_logs[0]["new_kv_tokens"] + sum(
    log["new_kv_tokens"] for log in cache_logs[1:]
)

print("no-cache recomputed KV tokens:", no_cache_recomputed)
print("with-cache newly projected KV tokens:", with_cache_new)
print()
print("per-step no-cache logs")
for log in no_cache_logs:
    print(log)
print()
print("per-step cache logs")
for log in cache_logs:
    print(log)

no-cache recomputed KV tokens: 15
with-cache newly projected KV tokens: 7

per-step no-cache logs
{'step': 1, 'sequence_len': 4, 'recomputed_kv_tokens': 4, 'last_query_key_count': 4, 'predicted_token': 'F'}
{'step': 2, 'sequence_len': 5, 'recomputed_kv_tokens': 5, 'last_query_key_count': 5, 'predicted_token': 'E'}
{'step': 3, 'sequence_len': 6, 'recomputed_kv_tokens': 6, 'last_query_key_count': 6, 'predicted_token': 'F'}

per-step cache logs
{'step': 0, 'cache_len': 4, 'new_kv_tokens': 4, 'note': 'initial prefix is encoded once'}
{'step': 1, 'cache_len': 5, 'new_kv_tokens': 1, 'last_query_key_count': 4, 'predicted_token': 'F'}
{'step': 2, 'cache_len': 6, 'new_kv_tokens': 1, 'last_query_key_count': 5, 'predicted_token': 'E'}
{'step': 3, 'cache_len': 7, 'new_kv_tokens': 1, 'last_query_key_count': 6, 'predicted_token': 'F'}


## 5. なぜ速くなるのか

系列長を `T`、生成ステップ数を `N` とする。

- キャッシュなしでは、各ステップで長さ `T`, `T+1`, `T+2`, ... の prefix 全体に対して `K`, `V` を作り直す
- キャッシュありでは、最初に長さ `T` の prefix を1回だけエンコードし、その後は各ステップで新規 token 1個分の `K`, `V` だけ作る

このため、特に長文を1 tokenずつ生成するときに差が大きくなる。

ただし、attention 自体は最新 `Q` が過去全部の `K` を見るので、参照先がゼロになるわけではない。省略されるのは主に**過去 token の `K`, `V` の再投影**である。

In [11]:
def total_projected_tokens_without_cache(prefix_len, steps):
    return sum(prefix_len + i for i in range(steps))


def total_projected_tokens_with_cache(prefix_len, steps):
    return prefix_len + steps


for prefix_len, steps in [(4, 3), (128, 32), (1024, 128)]:
    a = total_projected_tokens_without_cache(prefix_len, steps)
    b = total_projected_tokens_with_cache(prefix_len, steps)
    print(
        f"prefix={prefix_len:>4}, steps={steps:>3} -> without cache: {a:>6}, with cache: {b:>6}, ratio: {a / b:>8.2f}"
    )

prefix=   4, steps=  3 -> without cache:     15, with cache:      7, ratio:     2.14
prefix= 128, steps= 32 -> without cache:   4592, with cache:    160, ratio:    28.70
prefix=1024, steps=128 -> without cache: 139200, with cache:   1152, ratio:   120.83


## 6. メモリとのトレードオフ

KVキャッシュは高速化の代わりにメモリを使う。

- 各層で `K_cache` と `V_cache` を保持する必要がある
- 系列長が伸びるほどキャッシュも大きくなる
- multi-head では head ごとに保持するため、実運用ではさらに大きい

つまり、KVキャッシュは「計算を減らす代わりに、過去の表現をメモリに置いておく」方式だと捉えるとよい。

## 7. よくある誤解

### 誤解1: `Q` も全部キャッシュするのか？
通常の自己回帰生成では、最新 token の `Q` だけあれば次 token を出せるので、主役は `K` と `V` のキャッシュである。

### 誤解2: 学習時も同じように効くのか？
学習時は系列全体を並列に処理することが多く、推論時の1 tokenずつ生成とは計算パターンが違う。KVキャッシュは主に推論最適化である。

### 誤解3: prefix が変わっても同じキャッシュを使えるのか？
使えない。prefix が変われば `K`, `V` も変わるため、その prefix 専用のキャッシュを作り直す必要がある。

### 誤解4: KVキャッシュで attention の参照先そのものが減るのか？
減らない。最新 `Q` は依然として過去すべての `K` を見る。減るのは、過去 token 分の再計算である。

## 8. 確認問題

1. KVキャッシュで保存しているのは何か。
2. KVキャッシュありでも、各ステップで最低限再計算が必要なものは何か。
3. KVキャッシュが特に効くのは、どのような推論シナリオか。
4. KVキャッシュの主なデメリットは何か。

## 回答例

1. 過去トークンから計算済みの `K` と `V`。
2. 最新 token 分の `Q` と、新しく追加された token 分の `K`, `V`。
3. 自己回帰に1 tokenずつ長く生成する推論。特に prefix が長いほど効きやすい。
4. 各層・各head・各過去token分のキャッシュを保持するため、メモリ消費が増えること。